In [1]:
import numpy as np
import GPy
import matplotlib.pyplot as plt
from ma.mtgp import MTGP, HR_SIGNALS, RR_SIGNALS
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib qt
pd.set_option('display.max_columns', None)
parti_no = 1
mtgp = MTGP(parti_no=parti_no)

 c:\Users\firat\Desktop\MA\ma\ma\gp_utils.py:185: RuntimeWarning:divide by zero encountered in scalar divide
 c:\Users\firat\Desktop\MA\ma\ma\mtgp.py:103: SettingWithCopyWarning:
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
 c:\Users\firat\Desktop\MA\ma\ma\mtgp.py:104: SettingWithCopyWarning:
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
 c:\Users\firat\Desktop\MA\ma\ma\mtgp.py:105: SettingWithCopyWarning:
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the document

In [2]:
df_hr_train_prev = mtgp.hr_fused.loc[1053.5:1084]
df_hr_train = mtgp.hr_fused.loc[1103:1275]
df_hr_rest = mtgp.hr_fused.loc[1277:]

df_hr_ma = mtgp.hr_fused.loc[1083.5:1104]
df_hr_ma2 = mtgp.hr_fused.loc[1274.5:1277.5]

hr_ref = mtgp.hr_rr_ref["HR"]["hr"].loc[1053.5:]



fig, ax = plt.subplots()
ax.plot(df_hr_train.index, df_hr_train["hr_fused"], color = "green", linewidth = 3.0, marker = "o", label = "HR - Train")
ax.plot(df_hr_rest.index, df_hr_rest["hr_fused"], color = "green", linewidth = 3.0, marker = "o")
ax.plot(df_hr_train_prev.index, df_hr_train_prev["hr_fused"], color = "green", linewidth = 3.0, marker = "o")
ax.plot(df_hr_ma.index, df_hr_ma["hr_fused"], color = "red", linewidth = 1.0, marker = "o", label= "HR - MA")
ax.plot(df_hr_ma2.index, df_hr_ma2["hr_fused"], color = "red", linewidth = 1.0, marker = "o")

ax.plot(hr_ref.index, hr_ref, color = "orange", label = "HR - REF")

ax.grid()
ax.legend()

In [3]:
time_prev = df_hr_train_prev.index.to_numpy()[:, None]
time_hr = df_hr_train.index.to_numpy()[:, None]
time_rest = df_hr_rest.index.to_numpy()[:, None]
hr_prev = df_hr_train_prev["hr_fused"].to_numpy()[:, None]
hr_data = df_hr_train["hr_fused"].to_numpy()[:, None]
hr_rest = df_hr_rest["hr_fused"].to_numpy()[:, None]

time_hr = np.concatenate([time_prev, time_hr, time_rest])
hr_data = np.concatenate([hr_prev, hr_data, hr_rest])

# Combine RBF and Periodic kernels for more complex patterns
# rbf_kernel = GPy.kern.RBF(input_dim=1, variance=10.0, lengthscale=5.0)
# periodic_kernel = GPy.kern.PeriodicExponential(input_dim=1, variance=10.0, lengthscale=1.0, period=10)
# kernel = rbf_kernel + periodic_kernel
rbf_kernel = GPy.kern.RBF(input_dim=1, variance=10.0, lengthscale=5.0)
periodic_kernel_respiration = GPy.kern.PeriodicExponential(input_dim=1, variance=10.0, lengthscale=1.0, period=1.0)
periodic_kernel_mayer = GPy.kern.PeriodicExponential(input_dim=1, variance=10.0, lengthscale=1.0, period=10.0)

# Combine kernels
combined_kernel = rbf_kernel + periodic_kernel_respiration + periodic_kernel_mayer

gp_model = GPy.models.GPRegression(time_hr, hr_data, kernel=combined_kernel, normalizer=True)
gp_model.optimize()

In [4]:
time0 = df_hr_ma.index.to_numpy()[:, None]
time1 = df_hr_train.index.to_numpy()[:, None]
time2 = df_hr_ma2.index.to_numpy()[:,None]
time3 = df_hr_rest.index.to_numpy()[:, None]
time_test_hr = np.concatenate([time_prev, time0, time1, time2, time3])

# Prepare input matrices for each task
X_test_hr = np.hstack([time_test_hr, np.zeros_like(time_test_hr)])  # Task label 0

# Predict for each task
Y_pred_hr, Y_var_hr = gp_model.predict(X_test_hr)

# Plot results
# Heart Rate predictions
ax.plot(time_test_hr, Y_pred_hr, 'bx-', label='Predicted HR')
ax.fill_between(
    time_test_hr.flatten(),
    Y_pred_hr.flatten() - 1.96 * np.sqrt(Y_var_hr.flatten()),
    Y_pred_hr.flatten() + 1.96 * np.sqrt(Y_var_hr.flatten()),
    color="blue",
    alpha=0.2,
)

-------------
OLD

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RationalQuadratic,
    ConstantKernel as C,
    RBF,
    ExpSineSquared,
    Matern,
)
from sklearn.preprocessing import StandardScaler


kernel_se = RBF(length_scale=1.0, length_scale_bounds=(1e-6, 1e6))

kernel_respiration = ExpSineSquared(length_scale=1.0, periodicity=1.0)
kernel_mayer = ExpSineSquared(length_scale=1.0, periodicity=10.0)
combined_kernel = kernel_respiration + kernel_mayer + kernel_se
old_model = GaussianProcessRegressor(
    kernel=combined_kernel, n_restarts_optimizer=10, optimizer="fmin_l_bfgs_b"
)

time_prev = df_hr_train_prev.index.to_numpy()[:, None]
time_hr = df_hr_train.index.to_numpy()[:, None]
time_rest = df_hr_rest.index.to_numpy()[:, None]
hr_prev = df_hr_train_prev["hr_fused"].to_numpy()[:, None]
hr_data = df_hr_train["hr_fused"].to_numpy()[:, None]
hr_rest = df_hr_rest["hr_fused"].to_numpy()[:, None]

time_hr = np.concatenate([time_prev, time_hr, time_rest])
hr_data = np.concatenate([hr_prev, hr_data, hr_rest])
time_test_hr = np.concatenate([time_prev, time0, time1, time2, time3])

fig, ax = plt.subplots()
ax.plot(df_hr_train.index, df_hr_train["hr_fused"], color = "green", linewidth = 3.0, marker = "o", label = "HR - Train")
ax.plot(df_hr_rest.index, df_hr_rest["hr_fused"], color = "green", linewidth = 3.0, marker = "o")
ax.plot(df_hr_train_prev.index, df_hr_train_prev["hr_fused"], color = "green", linewidth = 3.0, marker = "o")
ax.plot(df_hr_ma.index, df_hr_ma["hr_fused"], color = "red", linewidth = 1.0, marker = "o", label= "HR - MA")
ax.plot(df_hr_ma2.index, df_hr_ma2["hr_fused"], color = "red", linewidth = 1.0, marker = "o")

ax.plot(hr_ref.index, hr_ref, color = "orange", label = "HR - REF")

ax.grid()
ax.legend()

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(time_hr)
y_train_scaled = scaler_y.fit_transform(hr_data).ravel()

old_model.fit(X_train_scaled, y_train_scaled)

x_scaled = scaler_X.transform(time_test_hr)
y_pred_scaled, sigma = old_model.predict(x_scaled, return_std=True)
y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

# Plot results
# Heart Rate predictions
ax.plot(time_test_hr, y_pred, 'bx-', label='Predicted HR')
ax.fill_between(
    time_test_hr.flatten(),
    y_pred.flatten() - 1.96 * np.sqrt(sigma.flatten()),
    y_pred.flatten() + 1.96 * np.sqrt(sigma.flatten()),
    color="blue",
    alpha=0.2,
)